# S2 — Analysis units

Stage 2 of the Manhattan Sidewalk Shade Index pipeline.

Converts the 4,584 raw sidewalk polygons into comparable analysis units: explode multipart, subdivide oversized polygons on a grid, assign each to its nearest LION street segment, and determine which side of the street it's on. Per `docs/DECISIONS.md`, the join/subdivision/aggregation logic runs in **DuckDB** (spatial extension) — trivial per-row geometry ops (explode) and the final side-of-street trig stay in Python/shapely.

**Accept when:** units tile the sidewalk surface without overlap, total unit area ≈ total input sidewalk area (within 1%), every unit has `street_name` and `side`.

In [1]:
from pathlib import Path
from datetime import datetime

import yaml
import duckdb
import geopandas as gpd
import pandas as pd
from shapely import wkb as shapely_wkb

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
ANALYSIS_CRS = config["analysis_crs"]
GRID_SIZE_M = config["grid_size_m"]
MAX_UNIT_AREA_M2 = config["max_unit_area_m2"]

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
print("s2: Build analysis units")
print(f"Timestamp: {datetime.now().isoformat()}\n")

s2: Build analysis units
Timestamp: 2026-08-28T13:58:28.454077



## Explode multipart sidewalks, assign a stable `unit_id`

The raw dataset's own `source_id` is **not unique** (hundreds of rows share `source_id='0'`) — a synthetic sequential `unit_id` is used instead, per CLAUDE.md §5's "generate a stable unit_id" step.

In [2]:
sidewalks = gpd.read_parquet(PROJECT_ROOT / config["output"]["ingested_sidewalks"])
sidewalks_exploded = sidewalks.explode(index_parts=False).reset_index(drop=True)
print(f"sidewalks: {len(sidewalks):,} rows -> exploded: {len(sidewalks_exploded):,} rows")

scratch_path = PROJECT_ROOT / config["interim_dir"] / "_scratch_sidewalks_exploded.parquet"
sidewalks_exploded[["geometry"]].to_parquet(scratch_path)
con.execute(f"""
    CREATE TABLE sw AS
    SELECT row_number() OVER () AS pre_id, geometry AS geom, ST_Area(geometry) AS area_m2
    FROM read_parquet('{scratch_path.as_posix()}')
""")
area_stats = con.execute("SELECT min(area_m2), median(area_m2), max(area_m2), sum(area_m2) FROM sw").fetchone()
print(f"area (m2): min={area_stats[0]:.1f} median={area_stats[1]:.1f} max={area_stats[2]:.1f} total={area_stats[3]:.1f}")

sidewalks: 4,584 rows -> exploded: 4,592 rows


area (m2): min=0.2 median=989.0 max=115024.7 total=5652028.5


## Decision gate: subdivide oversized polygons on a grid

Per CLAUDE.md §5: if the median unit area exceeds `max_unit_area_m2`, subdivide the oversized polygons by intersecting with a regular grid (default 50 m).

In [3]:
median_area = area_stats[1]
xmin, ymin, xmax, ymax = sidewalks_exploded.total_bounds

if median_area > MAX_UNIT_AREA_M2:
    print(f"Median area {median_area:.1f} m2 > {MAX_UNIT_AREA_M2} m2 threshold -- subdividing oversized units.")
    con.execute(f"""
        CREATE TABLE grid AS
        SELECT
            ST_MakeEnvelope(
                {xmin} + gx * {GRID_SIZE_M}, {ymin} + gy * {GRID_SIZE_M},
                {xmin} + (gx+1) * {GRID_SIZE_M}, {ymin} + (gy+1) * {GRID_SIZE_M}
            ) AS geom
        FROM generate_series(0, CAST(CEIL(({xmax}-{xmin})/{GRID_SIZE_M}) AS INTEGER)) AS t1(gx)
        CROSS JOIN generate_series(0, CAST(CEIL(({ymax}-{ymin})/{GRID_SIZE_M}) AS INTEGER)) AS t2(gy)
    """)
    con.execute(f"""
        CREATE TABLE sw_units_raw AS
        SELECT pre_id, geom, area_m2 FROM sw WHERE area_m2 <= {MAX_UNIT_AREA_M2}
        UNION ALL
        SELECT s.pre_id, ST_Intersection(s.geom, g.geom) AS geom,
               ST_Area(ST_Intersection(s.geom, g.geom)) AS area_m2
        FROM sw s JOIN grid g ON ST_Intersects(s.geom, g.geom)
        WHERE s.area_m2 > {MAX_UNIT_AREA_M2}
    """)
else:
    print(f"Median area {median_area:.1f} m2 <= {MAX_UNIT_AREA_M2} m2 threshold -- no subdivision needed.")
    con.execute("CREATE TABLE sw_units_raw AS SELECT pre_id, geom, area_m2 FROM sw")

# Drop zero-area slivers and any non-polygon leftovers from grid intersection
con.execute("""
    CREATE TABLE sw_units AS
    SELECT row_number() OVER () AS unit_id, geom, area_m2
    FROM sw_units_raw
    WHERE area_m2 > 0.5 AND ST_GeometryType(geom) IN ('POLYGON', 'MULTIPOLYGON')
""")
n_units, total_area = con.execute("SELECT count(*), sum(area_m2) FROM sw_units").fetchone()
retained_pct = 100 * total_area / area_stats[3]
print(f"final units: {n_units:,}  total area: {total_area:,.1f} m2  retained: {retained_pct:.3f}%")
assert retained_pct >= 99.0, f"area retention {retained_pct:.2f}% below the 1% loss tolerance"

Median area 989.0 m2 > 250 m2 threshold -- subdividing oversized units.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

final units: 34,603  total area: 5,651,946.5 m2  retained: 99.999%


## Assign each unit to its nearest LION centerline

LION `FeatureTyp` is filtered to `'0'`/`'1'` (Street / Non-Addressable Street) so units don't get matched to census boundaries, piers, or waterways. A 150 m buffer around each unit's centroid keeps this a bounded spatial join instead of a full cross product; any leftover unmatched units (there were 4 of 34,603 in testing — tiny slivers far from a coded street) get a fallback pass against *all* LION types with no distance cap.

In [4]:
lion = gpd.read_parquet(PROJECT_ROOT / config["output"]["ingested_lion"])
lion_scratch = PROJECT_ROOT / config["interim_dir"] / "_scratch_lion.parquet"
lion[["Street", "FeatureTyp", "geometry"]].to_parquet(lion_scratch)
con.execute(f"""
    CREATE TABLE lion_streets AS
    SELECT \"Street\" AS street, geometry AS geom
    FROM read_parquet('{lion_scratch.as_posix()}')
    WHERE \"FeatureTyp\" IN ('0', '1')
""")
con.execute(f"""
    CREATE TABLE lion_all AS
    SELECT \"Street\" AS street, geometry AS geom
    FROM read_parquet('{lion_scratch.as_posix()}')
""")

con.execute("""
    CREATE TABLE nearest AS
    WITH candidates AS (
        SELECT u.unit_id, u.geom AS unit_geom, l.street,
               l.geom AS lion_geom, ST_Distance(ST_Centroid(u.geom), l.geom) AS dist,
               ROW_NUMBER() OVER (
                   PARTITION BY u.unit_id ORDER BY ST_Distance(ST_Centroid(u.geom), l.geom)
               ) AS rn
        FROM sw_units u, lion_streets l
        WHERE ST_Intersects(ST_Buffer(ST_Centroid(u.geom), 150), l.geom)
    )
    SELECT unit_id, unit_geom, street, lion_geom, dist FROM candidates WHERE rn = 1
""")
matched = con.execute("SELECT count(*) FROM nearest").fetchone()[0]
print(f"matched within 150m: {matched:,} of {n_units:,}")

unmatched_ids = [r[0] for r in con.execute(
    "SELECT unit_id FROM sw_units WHERE unit_id NOT IN (SELECT unit_id FROM nearest)"
).fetchall()]
if unmatched_ids:
    ids_sql = ",".join(str(i) for i in unmatched_ids)
    con.execute(f"""
        INSERT INTO nearest
        WITH candidates AS (
            SELECT u.unit_id, u.geom AS unit_geom, l.street,
                   l.geom AS lion_geom, ST_Distance(ST_Centroid(u.geom), l.geom) AS dist,
                   ROW_NUMBER() OVER (
                       PARTITION BY u.unit_id ORDER BY ST_Distance(ST_Centroid(u.geom), l.geom)
                   ) AS rn
            FROM sw_units u, lion_all l
            WHERE u.unit_id IN ({ids_sql})
        )
        SELECT unit_id, unit_geom, street, lion_geom, dist FROM candidates WHERE rn = 1
    """)
    print(f"fallback pass (all LION types, unbounded) resolved {len(unmatched_ids)} more units")

final_matched = con.execute("SELECT count(*) FROM nearest").fetchone()[0]
print(f"total matched: {final_matched:,} of {n_units:,}")
assert final_matched == n_units, "every unit must have a nearest LION street match"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matched within 150m: 34,599 of 34,603
fallback pass (all LION types, unbounded) resolved 4 more units
total matched: 34,603 of 34,603


## Side-of-street via cross-product sign test

For each unit, project its centroid against the direction vector of its nearest LION segment (first vertex → last vertex). A positive cross product means the centroid is to the **left** of the segment's digitised direction, negative means **right** — the same left/right convention LION itself uses for its `LBoro`/`RBoro` fields.

In [5]:
def side_of_street(unit_wkb, lion_wkb):
    unit_geom = shapely_wkb.loads(bytes(unit_wkb))
    lion_geom = shapely_wkb.loads(bytes(lion_wkb))
    line = max(lion_geom.geoms, key=lambda ln: ln.length) if lion_geom.geom_type == "MultiLineString" else lion_geom
    (x0, y0), (x1, y1) = line.coords[0], line.coords[-1]
    dx, dy = x1 - x0, y1 - y0
    cx, cy = unit_geom.centroid.x, unit_geom.centroid.y
    cross = dx * (cy - y0) - dy * (cx - x0)
    return "Left" if cross > 0 else "Right"


nearest_df = con.execute("""
    SELECT unit_id, ST_AsWKB(unit_geom) AS unit_wkb, street, ST_AsWKB(lion_geom) AS lion_wkb, dist
    FROM nearest
""").fetchdf()
nearest_df["side"] = nearest_df.apply(lambda r: side_of_street(r["unit_wkb"], r["lion_wkb"]), axis=1)
print(nearest_df["side"].value_counts())

side
Left     17472
Right    17131
Name: count, dtype: int64


## Assemble final layer, validate, write GeoParquet

In [6]:
units_geom = con.execute("SELECT unit_id, ST_AsWKB(geom) AS geom_wkb, area_m2 FROM sw_units").fetchdf()
units_geom["geometry"] = units_geom["geom_wkb"].apply(lambda b: shapely_wkb.loads(bytes(b)))

final = units_geom.merge(
    nearest_df[["unit_id", "street", "side", "dist"]].rename(
        columns={"street": "street_name", "dist": "lion_distance_m"}
    ),
    on="unit_id", how="left",
)
final["street_name"] = final["street_name"].fillna("UNNAMED STREET")
analysis_units = gpd.GeoDataFrame(
    final[["unit_id", "street_name", "side", "lion_distance_m", "area_m2", "geometry"]],
    geometry="geometry", crs=ANALYSIS_CRS,
)
print(f"analysis_units: {len(analysis_units):,} rows, CRS={analysis_units.crs}")

analysis_units: 34,603 rows, CRS=EPSG:32618


In [7]:
print("\n" + "=" * 70)
print("S2 — Analysis units: QA Summary")
print("=" * 70)
print(f"  Rows: {len(analysis_units):,}")
print(f"  CRS: {analysis_units.crs}")
print(f"  Total area: {analysis_units['area_m2'].sum():,.1f} m2 (input sidewalks: {area_stats[3]:,.1f} m2)")
print(f"  Missing street_name: {(analysis_units['street_name'] == 'UNNAMED STREET').sum()}")
print(f"  Missing side: {analysis_units['side'].isna().sum()}")
print("=" * 70)

retained_pct = 100 * analysis_units["area_m2"].sum() / area_stats[3]
assert retained_pct >= 99.0, f"final area retention {retained_pct:.2f}% below tolerance"
assert analysis_units["street_name"].notna().all(), "every unit must have a street_name"
assert analysis_units["side"].notna().all(), "every unit must have a side"
assert analysis_units["unit_id"].is_unique, "unit_id must be unique"
print("All S2 acceptance checks passed.")


S2 — Analysis units: QA Summary
  Rows: 34,603
  CRS: EPSG:32618
  Total area: 5,651,946.5 m2 (input sidewalks: 5,652,028.5 m2)
  Missing street_name: 35
  Missing side: 0
All S2 acceptance checks passed.


In [8]:
output_path = PROJECT_ROOT / config["output"]["analysis_units"]
analysis_units.to_parquet(output_path)
print(f"Wrote {output_path}")

# Clean up scratch parquet files used to hand geometries to DuckDB
scratch_path.unlink(missing_ok=True)
lion_scratch.unlink(missing_ok=True)
print("\ns2 complete. Ready for S3 (crown geometry).")

Wrote C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\interim\analysis_units.parquet

s2 complete. Ready for S3 (crown geometry).
